# TTS content-error verifier — GPU steps (Colab T4)

Runtime → Change runtime type → **T4 GPU**. Outputs are saved to Google Drive so a disconnect doesn't lose work. Every script is resumable: just re-run the cell.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code and persist outputs to Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/JagadishSivakumar/tts-content-verifier.git /content/repo  # or upload the zip and unzip to /content/repo
%cd /content/repo
!mkdir -p /content/drive/MyDrive/tts-content-verifier/outputs
!rm -rf outputs && ln -s /content/drive/MyDrive/tts-content-verifier/outputs outputs

## 2. Install (record exact versions)

In [ ]:
!pip install -q -r requirements-gpu.txt
!pip freeze > outputs/env.txt
!python -m unittest discover tests

## 3. Smoke test: 3 prompts × 2 models. **Listen before the full run.**

In [ ]:
!python scripts/01_make_prompts.py
!python scripts/02_generate.py --limit 3

In [ ]:
import json, IPython.display as d
for line in open('outputs/generations.jsonl'):
    g = json.loads(line)
    print(g['clip_id'], f"{g['duration_s']:.1f}s")
    d.display(d.Audio('outputs/' + g['wav']))

## 4. Full generation (200 clips) and transcription

In [ ]:
!python scripts/02_generate.py
!python scripts/03_transcribe.py

## 5. Build the blind labelling pages, then download

Download `labeler.zip`, unzip it on your laptop and open `primary.html` in Chrome. Don't look at `transcripts.jsonl` until labelling is finished.

In [ ]:
!python scripts/04_build_labeler.py
!cd outputs && zip -qr labeler.zip labeler && ls -lh labeler.zip

## 6. After labelling: upload `labels/primary.csv` and `labels/second.csv`, then evaluate and run the intervention

In [ ]:
from google.colab import files
import shutil, os
os.makedirs('labels', exist_ok=True)
for name in files.upload():
    shutil.move(name, f'labels/{name}')
!python scripts/05_evaluate.py

In [ ]:
!python scripts/06_regenerate.py --run
!python scripts/04_build_labeler.py --source regen
!cd outputs && zip -qr labeler_regen.zip labeler

## 7. After labelling the regenerated clips (append rows to `labels/primary.csv` and re-upload)

In [ ]:
!python scripts/06_regenerate.py --evaluate